In [0]:
bronze = spark.table("workspace.api_logs_schema.bronze_api_logs_raw")

In [0]:
#print("Bronze count: ",bronze.count())
#display(bronze.limit(5))

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import from_json, col, to_timestamp, to_date, hour
from pyspark.sql.functions import count, countDistinct
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
schema = StructType([
    StructField("request_id",StringType()),
    StructField("event_time",StringType()),
    StructField("service",StringType()),
    StructField("endpoint",StringType()),
    StructField("method",StringType()),
    StructField("status_code",IntegerType()),
    StructField("latency_ms",IntegerType()),
    StructField("bytes_in",IntegerType()),
    StructField("bytes_out",IntegerType()),
    StructField("client_type",StringType()),
    StructField("region",StringType()),
    StructField("host",StringType())
])

In [0]:
parsed = bronze.withColumn('data', from_json(col('value'),schema))

In [0]:
#print("Parsed count: ", parsed.count())
#display(parsed.limit(3))

In [0]:
malformed = parsed.filter(col("data").isNull())


In [0]:
#print("Malformed rows:", malformed.count())


In [0]:
clean = parsed.filter(col("data").isNotNull()).select("data.*")

In [0]:
#print("parsable rows count: ", clean.count())
#display(clean.limit(3))

parsable rows count:  1280995


In [0]:
clean.select(
    count("*"),
    countDistinct("request_id")
).show()

+--------+--------------------------+
|count(1)|count(DISTINCT request_id)|
+--------+--------------------------+
| 1280995|                    624986|
+--------+--------------------------+



In [0]:
dupes = clean.groupBy("request_id") \
    .count() \
    .filter(col("count") > 1) \
    .orderBy(col("count").desc())

#display(dupes.limit(20))

request_id,count
null,6435
req_20260428_000703,7
req_20260515_000165,5
req_20260517_008459,5
req_20260519_000637,5
req_20260505_000012,5
req_20260521_000296,5
req_20260508_000007,5
req_20260429_000151,5
req_20260512_002327,5


In [0]:
'''display(
    clean.filter(col("request_id") == "req_20260428_000703")
)'''

request_id,event_time,service,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host
req_20260428_000703,2026-04-28T09:24:52Z,search-api,/v1/search,DELETE,200,132,1202,2115,mobile,eu-west-1,app-02
req_20260428_000703,2026-04-28T09:24:52Z,search-api,/v1/search,DELETE,200,132,1202,2115,mobile,eu-west-1,app-02
req_20260428_000703,2026-04-28T09:24:52Z,search-api,/v1/search,DELETE,200,132,1202,2115,mobile,eu-west-1,app-02
req_20260428_000703,2026-04-28T09:24:52Z,search-api,/v1/search,DELETE,200,132,1202,2115,mobile,eu-west-1,app-02
req_20260428_000703,2026-04-28T09:24:52Z,search-api,/v1/search,DELETE,200,132,1202,2115,mobile,eu-west-1,app-02
req_20260428_000703,2026-04-28T10:55:51Z,search-api,/v1/search,DELETE,201,190,1683,10810,mobile,ap-south-1,app-01
req_20260428_000703,2026-04-28T10:55:51Z,search-api,/v1/search,DELETE,201,190,1683,10810,mobile,ap-south-1,app-01


In [0]:
valid = clean.filter(
    (col("request_id").isNotNull()) &
    (col("status_code").between(100, 599)) &
    (col("latency_ms") >= 0)
)


In [0]:
#print("Valid rows:", valid.count())


Valid rows: 1261460


In [0]:

invalid = clean.filter(
    (col("request_id").isNull()) |
    (~col("status_code").between(100, 599)) |
    (col("latency_ms") < 0)
)

In [0]:
#print("Invalid rows:", invalid.count())

#display(invalid.limit(10))

In [0]:
dedupe_keys = [
    "request_id",
    "event_time",
    "service",
    "endpoint",
    "method",
    "status_code",
    "latency_ms",
    "bytes_in",
    "bytes_out",
    "client_type",
    "region",
    "host"
]

window = Window.partitionBy(*dedupe_keys).orderBy("event_time")

deduped = valid \
    .withColumn("rn", row_number().over(window)) \
    .filter(col("rn") == 1) \
    .drop("rn")

In [0]:
'''valid_count = valid.count()
deduped_count = deduped.count()

print("Valid rows:", valid_count)
print("Deduped rows:", deduped_count)
print("Rows removed as true duplicates:", valid_count - deduped_count)

display(deduped.limit(5))'''

Valid rows: 1261460
Deduped rows: 1230948
Rows removed as true duplicates: 30512


request_id,event_time,service,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host
req_20260428_000012,2026-04-28T10:04:02Z,notifications-api,/v1/notify,PUT,401,50,1403,18724,desktop,ap-south-1,app-03
req_20260428_000015,2026-04-28T10:56:44Z,auth-api,/v1/logout,PUT,400,64,1822,42465,mobile,us-east-1,app-03
req_20260428_000019,2026-04-28T09:48:41Z,notifications-api,/v1/notify,DELETE,201,83,1941,24783,desktop,us-east-1,app-04
req_20260428_000025,2026-04-28T09:57:42Z,auth-api,/v1/refresh,PUT,500,150,1599,32138,web,ap-south-1,app-04
req_20260428_000028,2026-04-28T09:00:19Z,auth-api,/v1/logout,DELETE,200,89,929,3251,web,us-east-1,app-04


In [0]:
silver_clean = deduped \
    .withColumn("event_ts", to_timestamp("event_time")) \
    .withColumn("event_date", to_date("event_ts")) \
    .withColumn("event_hour", hour("event_ts")) \
    .withColumn("is_client_error", ((col("status_code") >= 400) & (col("status_code") < 500)).cast("int")) \
    .withColumn("is_server_error", (col("status_code") >= 500).cast("int"))

display(silver_clean.limit(5))

request_id,event_time,service,endpoint,method,status_code,latency_ms,bytes_in,bytes_out,client_type,region,host,event_ts,event_date,event_hour,is_client_error,is_server_error
req_20260428_000012,2026-04-28T10:04:02Z,notifications-api,/v1/notify,PUT,401,50,1403,18724,desktop,ap-south-1,app-03,2026-04-28T10:04:02.000Z,2026-04-28,10,1,0
req_20260428_000015,2026-04-28T10:56:44Z,auth-api,/v1/logout,PUT,400,64,1822,42465,mobile,us-east-1,app-03,2026-04-28T10:56:44.000Z,2026-04-28,10,1,0
req_20260428_000019,2026-04-28T09:48:41Z,notifications-api,/v1/notify,DELETE,201,83,1941,24783,desktop,us-east-1,app-04,2026-04-28T09:48:41.000Z,2026-04-28,9,0,0
req_20260428_000025,2026-04-28T09:57:42Z,auth-api,/v1/refresh,PUT,500,150,1599,32138,web,ap-south-1,app-04,2026-04-28T09:57:42.000Z,2026-04-28,9,0,1
req_20260428_000028,2026-04-28T09:00:19Z,auth-api,/v1/logout,DELETE,200,89,929,3251,web,us-east-1,app-04,2026-04-28T09:00:19.000Z,2026-04-28,9,0,0


In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.api_logs_schema.silver_api_logs_clean")

silver_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.api_logs_schema.silver_api_logs_clean")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace.api_logs_schema.silver_api_logs_quarantine")

invalid.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.api_logs_schema.silver_api_logs_quarantine")

In [0]:
'''print("Bronze rows:", spark.table("workspace.api_logs_schema.bronze_api_logs_raw").count())
print("Silver clean rows:", spark.table("workspace.api_logs_schema.silver_api_logs_clean").count())
print("Silver quarantine rows:", spark.table("workspace.api_logs_schema.silver_api_logs_quarantine").count())'''

Bronze rows: 1280995
Silver clean rows: 1230948
Silver quarantine rows: 19535


In [0]:
print("Bronze rows:", spark.table("workspace.api_logs_schema.bronze_api_logs_raw").count())
print("Silver clean rows:", spark.table("workspace.api_logs_schema.silver_api_logs_clean").count())
print("Silver quarantine rows:", spark.table("workspace.api_logs_schema.silver_api_logs_quarantine").count())

print("dbt staging rows:", spark.table("workspace.gold_dbt.stg_api_logs_clean").count())
print("dbt service daily rows:", spark.table("workspace.gold_dbt.fct_service_daily_kpis").count())
print("dbt endpoint hourly rows:", spark.table("workspace.gold_dbt.fct_endpoint_hourly_kpis").count())

Bronze rows: 1280995
Silver clean rows: 1230948
Silver quarantine rows: 19535
dbt staging rows: 1230948
dbt service daily rows: 100
dbt endpoint hourly rows: 350
